In [2]:
# ========================
# PARTE 0: MONTAR DRIVE E INSTALAR DEPENDÊNCIAS
# ========================
from google.colab import drive
drive.mount('/content/drive')

!apt-get update > /dev/null
!apt install chromium-chromedriver -y > /dev/null || true
!cp /usr/lib/chromium-browser/chromedriver /usr/bin || true
!pip install selenium pandas openpyxl requests > /dev/null

# ========================
# PARTE 1: IMPORTS E CONFIGS
# ========================
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
import pandas as pd
import os, time, requests, zipfile, re
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor

# Configurações do navegador
options = Options()
options.binary_location = "/usr/bin/chromium-browser"
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--blink-settings=imagesEnabled=false")
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 7)

# ========================
# PARTE 2: CONFIG ALGOLIA
# ========================
ALGOLIA_APP_ID = "1CF3ZT43ZU"
ALGOLIA_API_KEY = "150c68d1c61fc1835826a57a203dab72"
ALGOLIA_INDEX = "production_products"

headers_algolia = {
    "Content-Type": "application/json",
    "X-Algolia-Application-Id": ALGOLIA_APP_ID,
    "X-Algolia-API-Key": ALGOLIA_API_KEY,
}

termos_busca = ["pisos", "revestimentos", "porcelanatos"]

print("🔍 Deseja coletar 100% ou 50% dos produtos disponíveis?")
while True:
    escolha = input("Digite '100' ou '50': ").strip()
    if escolha in ["100", "50"]:
        break
    else:
        print("❌ Escolha inválida. Tente novamente.")

fator_coleta = 1.0 if escolha == "100" else 0.5


def buscar_links_por_categoria(termos):
    links = set()
    for termo in termos:
        print(f"🔍 Buscando: {termo}")
        pagina = 0
        total_encontrado = None
        while True:
            url = f"https://{ALGOLIA_APP_ID}-dsn.algolia.net/1/indexes/{ALGOLIA_INDEX}/query"
            payload = { "params": f"query={termo}&page={pagina}&hitsPerPage=250" }
            res = requests.post(url, headers=headers_algolia, json=payload)
            if res.status_code != 200:
                break
            data = res.json()
            if total_encontrado is None:
                total_encontrado = int(data.get("nbHits", 0))
                limite_termo = int(total_encontrado * fator_coleta)
            hits = data.get("hits", [])
            if not hits:
                break
            for item in hits:
                relative_url = item.get("url")
                if relative_url:
                    full_url = f"https://www.leroymerlin.com.br{relative_url}" if not relative_url.startswith("http") else relative_url
                    links.add(full_url)
                    if len(links) >= limite_termo:
                        return list(links)
            pagina += 1
            if pagina >= data.get("nbPages", 0):
                break
    return list(links)

print("🔍 Coletando links...")
todos_links = buscar_links_por_categoria(termos_busca)
print(f"✅ Total de links coletados: {len(todos_links)}\n")

# ========================
# PARTE 3: EXTRAÇÃO DE PRODUTO
# ========================
def extrair_dados_produto(url):
    driver.get(url)
    try:
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "h1")))
    except TimeoutException:
        return {'URL': url, 'Erro': 'Timeout'}

    dados = {'URL': url}
    dados['Nome'] = driver.find_element(By.CSS_SELECTOR, "h1").text if driver.find_elements(By.CSS_SELECTOR, "h1") else None
    dados['Código'] = url.split("_")[-1] if "_" in url else None

    preco_txt = None
    seletores = [
        '[data-cy="product-price"]',
        '[data-cy="product-price-integer"]',
        '[data-cy="product-price-sale"]',
        '.price__sales', '.value__integer', '.price', '.price__value', '.sales-price', '.main-price'
    ]
    for sel in seletores:
        elems = driver.find_elements(By.CSS_SELECTOR, sel)
        if elems:
            text = elems[0].text.strip()
            if text and text.lower() != "grátis":
                preco_txt = text
                break

    if not preco_txt:
        html = driver.page_source
        match = re.search(r"R\$ ?\d{1,3}(?:\.\d{3})*,\d{2}", html)
        if match:
            preco_txt = match.group()

    if preco_txt:
        try:
            preco_val = float(preco_txt.replace("R$", "").replace("/m²", "").replace(" ", "").replace(".", "").replace(",", "."))
            dados['Preço Promocional'] = f"{preco_val:.2f}".replace(".", ",")
        except:
            dados['Preço Promocional'] = preco_txt
    else:
        dados['Preço Promocional'] = "N/A"

    dados['Descrição'] = driver.find_element(By.CSS_SELECTOR, 'meta[name="description"]').get_attribute("content") if driver.find_elements(By.CSS_SELECTOR, 'meta[name="description"]') else None
    dados['Imagem'] = driver.find_element(By.CSS_SELECTOR, 'meta[property="og:image"]').get_attribute("content") if driver.find_elements(By.CSS_SELECTOR, 'meta[property="og:image"]') else "N/A"

    specs = {}
    for tr in driver.find_elements(By.CSS_SELECTOR, 'table tr'):
        try:
            k = tr.find_element(By.CSS_SELECTOR, 'th').text.strip()
            v = tr.find_element(By.CSS_SELECTOR, 'td').text.strip()
            specs[k] = v
        except:
            continue
    dados['Especificações'] = "\n".join(f"{k}: {v}" for k, v in specs.items())
    dados['Loja'] = "Leroy Merlin"
    return dados

# ========================
# PARTE 4: COLETA + SALVAMENTO A CADA 100
# ========================
data_nome = datetime.now().strftime("%Y-%m-%d")
base_drive = f"/content/drive/MyDrive/WebScraping/FEITO/Leroy/COLETA_LEROY/{data_nome}"
pasta_imagens = os.path.join(base_drive, "imagens")
os.makedirs(pasta_imagens, exist_ok=True)

coletados = []

def salvar_parcial(contador):
    df = pd.DataFrame(coletados)
    df.drop(columns=[c for c in df.columns if c in ['Erro']], inplace=True, errors='ignore')
    colunas = ['Código', 'Nome', 'Preço Promocional', 'Descrição', 'Especificações', 'Imagem', 'URL', 'Loja']
    df = df[colunas]
    arquivo_csv = os.path.join(base_drive, f"leroymerelin_{data_nome}_{contador}.csv")
    df.to_csv(arquivo_csv, index=False, sep=";")
    print(f"💾 Parcial salva: {arquivo_csv}")

for i, link in enumerate(todos_links, 1):
    try:
        dados = extrair_dados_produto(link)
        codigo = dados.get("Código", f"produto_{i}")
        caminho_img = os.path.join(pasta_imagens, f"{codigo}.png")
        if not os.path.exists(caminho_img) and dados.get("Imagem") not in [None, "N/A"]:
            try:
                resp = requests.get(dados["Imagem"], headers={"User-Agent": "Mozilla/5.0"}, timeout=10, stream=True)
                if resp.status_code == 200:
                    with open(caminho_img, "wb") as f:
                        for cb in resp.iter_content(1024):
                            f.write(cb)
            except:
                pass
        coletados.append(dados)
        print(f"✅ {i}/{len(todos_links)} | {dados['Nome']}")
        if i % 100 == 0:
            salvar_parcial(i)
    except Exception as e:
        print(f"❌ Erro ao processar {link}: {e}")

salvar_parcial(len(todos_links))

# ========================
# ZIP FINAL DE IMAGENS
# ========================
zipname = os.path.join(base_drive, f"imagens_leroymerelin_{data_nome}.zip")
with zipfile.ZipFile(zipname, "w", zipfile.ZIP_DEFLATED) as z:
    for fn in os.listdir(pasta_imagens):
        z.write(os.path.join(pasta_imagens, fn), fn)

print(f"📦 Imagens zipadas em: {zipname}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


cp: '/usr/lib/chromium-browser/chromedriver' and '/usr/bin/chromedriver' are the same file
🔍 Deseja coletar 100% ou 50% dos produtos disponíveis?
Digite '100' ou '50': 50
🔍 Coletando links...
🔍 Buscando: pisos
🔍 Buscando: revestimentos
🔍 Buscando: porcelanatos
✅ Total de links coletados: 1984

✅ 1/1984 | Porcelanato Pedra Acetinado Borda Reta Externo 90x90x1cm Iseo Eliane
✅ 2/1984 | Porcelanato Pedra Acetinado Borda Reta Externo 72x72cm Basalto Beige Savane
✅ 3/1984 | Piso Cerâmico Pastilhado Brilhante Borda Arredondada Externo 34x34cm PPF34020 Incefra
✅ 4/1984 | Revestimento Marmorizado Brilhante 33x61cm Place Eliane
✅ 5/1984 | Piso Cerâmico Madeira Brilhante 57x57cm Cai